# Bulos Translator — LSTM Training (All 4 Models)

**Purpose:** Train direction-specific LSTM models for Bulos translation  
**Hardware:** GPU recommended (T4, P100, V100); CPU also works (slower)  
**Duration:** ~1–2 hours on GPU, ~7 hours on CPU  

## Models Trained
1. **Bulos → English** (`bul_en`)
2. **English → Bulos** (`en_bul`)
3. **Bulos → Filipino** (`bul_tl`)
4. **Filipino → Bulos** (`tl_bul`)

## Prerequisites
Run `scripts/prepare_lstm_training_data.py` once first to generate the
training split JSON files from `data/dataset.xlsx`.


In [ ]:
import json
import os
import time
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Tuple

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print('TensorFlow version:', tf.__version__)


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'\n✓ GPU AVAILABLE: {len(gpus)} device(s)')
    for i, gpu in enumerate(gpus):
        print(f'  GPU {i}: {gpu}')
        tf.config.experimental.set_memory_growth(gpu, True)
    DEVICE = tf.test.gpu_device_name() or '/device:GPU:0'
else:
    print('\n⚠  No GPU detected — training will use CPU (slower but works)')
    DEVICE = '/device:CPU:0'

print(f'\nCompute device: {DEVICE}')


In [ ]:
# ---------------------------------------------------------------------------
# Paths — relative to project root
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(globals().get('__file__', './')).resolve().parent
# When running in Jupyter the __file__ trick may not work;
# fall back to CWD (run the notebook from the project root).
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR   = PROJECT_ROOT / 'data' / 'lstm_training'
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'lstm'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------------------------
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 50
EMBEDDING_DIM  = 128
LSTM_UNITS     = 256
BATCH_SIZE     = 32
MAX_EPOCHS     = 500
PATIENCE       = 20

START_TOKEN = '<start>'
END_TOKEN   = '<end>'
PAD_TOKEN   = '<pad>'
UNK_TOKEN   = '<unk>'

# NOTE: uses 'tl' (Tagalog) not 'fil' to match the project's language codes
DIRECTIONS = [
    ('bul_en', 'Bulos → English'),
    ('en_bul', 'English → Bulos'),
    ('bul_tl', 'Bulos → Filipino'),
    ('tl_bul', 'Filipino → Bulos'),
]

print('Configuration:')
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  OUTPUT_DIR: {OUTPUT_DIR}')
print(f'  Max source length : {MAX_SOURCE_LEN}')
print(f'  Max target length : {MAX_TARGET_LEN}')
print(f'  Embedding dim     : {EMBEDDING_DIM}')
print(f'  LSTM units        : {LSTM_UNITS}')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Max epochs        : {MAX_EPOCHS}')
print(f'  Early stop patience: {PATIENCE}')


In [ ]:
print('Verifying training data...')
all_ok = True
for direction, label in DIRECTIONS:
    files = ['train.json', 'val.json', 'test.json', 'metadata.json']
    missing = [f for f in files if not (DATA_DIR / direction / f).exists()]
    if missing:
        print(f'  ✗ {direction}: missing {missing}')
        all_ok = False
    else:
        with open(DATA_DIR / direction / 'train.json', encoding='utf-8') as f:
            n = len(json.load(f))
        print(f'  ✓ {direction}: {n} training pairs')

if not all_ok:
    raise RuntimeError(
        'Training data not found.\n'
        'Run:  python scripts/prepare_lstm_training_data.py'
    )
print('\n✓ All training data present')


In [ ]:
def load_data(direction: str, split: str) -> List[Dict]:
    """Load train/val/test split for a direction."""
    file_path = DATA_DIR / direction / f'{split}.json'
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)


def prepare_tokenizers(train_pairs: List[Dict]) -> Tuple[Tokenizer, Tokenizer]:
    """
    Build tokenizers from TRAINING DATA ONLY.
    Source: character-level (preserves Bulos accents).
    Target: word-level.
    """
    sources      = [p['source'] for p in train_pairs]
    targets      = [p['target'] for p in train_pairs]
    target_vocab = targets + [START_TOKEN, END_TOKEN]

    src_tokenizer = Tokenizer(
        filters='', lower=False, oov_token=UNK_TOKEN, char_level=True
    )
    src_tokenizer.fit_on_texts(sources)

    tgt_tokenizer = Tokenizer(
        filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n',
        lower=True, oov_token=UNK_TOKEN
    )
    tgt_tokenizer.fit_on_texts(target_vocab)

    return src_tokenizer, tgt_tokenizer


def prepare_sequences(pairs, src_tokenizer, tgt_tokenizer):
    """Encode and pad encoder/decoder sequences."""
    sources = [p['source'] for p in pairs]
    targets = [p['target'] for p in pairs]

    enc_in = src_tokenizer.texts_to_sequences(sources)
    enc_in = pad_sequences(enc_in, maxlen=MAX_SOURCE_LEN,
                           padding='post', truncating='post')

    dec_in_texts  = [f'{START_TOKEN} {t}' for t in targets]
    dec_tgt_texts = [f'{t} {END_TOKEN}'   for t in targets]

    dec_in  = tgt_tokenizer.texts_to_sequences(dec_in_texts)
    dec_tgt = tgt_tokenizer.texts_to_sequences(dec_tgt_texts)

    dec_in  = pad_sequences(dec_in,  maxlen=MAX_TARGET_LEN,
                            padding='post', truncating='post')
    dec_tgt = pad_sequences(dec_tgt, maxlen=MAX_TARGET_LEN,
                            padding='post', truncating='post')

    return enc_in, dec_in, dec_tgt


def save_tokenizer(tokenizer: Tokenizer, path: Path):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(tokenizer.to_json())


print('✓ Helper functions defined')


In [ ]:
def create_model(src_vocab_size: int, tgt_vocab_size: int) -> keras.Model:
    """Encoder-decoder LSTM seq2seq model."""
    encoder_inputs    = layers.Input(shape=(None,), name='encoder_input')
    encoder_embedding = layers.Embedding(
        src_vocab_size, EMBEDDING_DIM, mask_zero=True, name='encoder_embedding'
    )(encoder_inputs)
    encoder_lstm = layers.LSTM(
        LSTM_UNITS, return_state=True, name='encoder_lstm'
    )
    _, state_h, state_c = encoder_lstm(encoder_embedding)

    decoder_inputs    = layers.Input(shape=(None,), name='decoder_input')
    decoder_embedding = layers.Embedding(
        tgt_vocab_size, EMBEDDING_DIM, name='decoder_embedding'
    )(decoder_inputs)
    decoder_lstm = layers.LSTM(
        LSTM_UNITS, return_sequences=True, return_state=True, name='decoder_lstm'
    )
    decoder_out, _, _ = decoder_lstm(
        decoder_embedding, initial_state=[state_h, state_c]
    )
    decoder_dense = layers.Dense(
        tgt_vocab_size, activation='softmax', name='decoder_dense'
    )
    decoder_out = decoder_dense(decoder_out)

    return keras.Model([encoder_inputs, decoder_inputs], decoder_out,
                       name='lstm_seq2seq')


def create_masked_loss():
    scce = keras.losses.SparseCategoricalCrossentropy(
        from_logits=False, reduction='none'
    )
    def masked_loss(y_true, y_pred):
        loss = scce(y_true, y_pred)
        mask = tf.cast(tf.not_equal(y_true, 0), dtype=loss.dtype)
        return tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)
    return masked_loss


def create_masked_accuracy():
    def masked_accuracy(y_true, y_pred):
        y_true_i = tf.cast(y_true, tf.int32)
        y_pred_i = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
        mask     = tf.cast(tf.not_equal(y_true_i, 0), tf.float32)
        matches  = tf.cast(tf.equal(y_true_i, y_pred_i), tf.float32) * mask
        return tf.reduce_sum(matches) / (tf.reduce_sum(mask) + 1e-9)
    return masked_accuracy


print('✓ Model architecture defined')


In [ ]:
def build_inference_models(model: keras.Model):
    """Split the trained model into standalone encoder and decoder."""
    encoder_inputs = model.input[0]
    _, state_h, state_c = model.get_layer('encoder_lstm').output
    encoder_model  = keras.Model(encoder_inputs, [state_h, state_c])

    decoder_inputs        = model.input[1]
    dec_state_h_in        = layers.Input(shape=(LSTM_UNITS,), name='dec_h_in')
    dec_state_c_in        = layers.Input(shape=(LSTM_UNITS,), name='dec_c_in')
    decoder_states_inputs = [dec_state_h_in, dec_state_c_in]

    dec_emb    = model.get_layer('decoder_embedding')
    dec_lstm   = model.get_layer('decoder_lstm')
    dec_dense  = model.get_layer('decoder_dense')

    dec_embedded          = dec_emb(decoder_inputs)
    dec_out, new_h, new_c = dec_lstm(
        dec_embedded, initial_state=decoder_states_inputs
    )
    dec_out    = dec_dense(dec_out)

    decoder_model = keras.Model(
        [decoder_inputs] + decoder_states_inputs,
        [dec_out, new_h, new_c]
    )
    return encoder_model, decoder_model


def decode_sequence(
    input_seq,
    encoder_model,
    decoder_model,
    tgt_tokenizer,
    max_length: int = MAX_TARGET_LEN,
) -> str:
    """Greedy autoregressive decoding."""
    states   = encoder_model.predict(input_seq, verbose=0)
    start_id = tgt_tokenizer.word_index.get(START_TOKEN)
    end_id   = tgt_tokenizer.word_index.get(END_TOKEN)
    if start_id is None or end_id is None:
        return ''

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = start_id
    decoded = []

    for _ in range(max_length):
        out_tokens, h, c = decoder_model.predict(
            [target_seq] + states, verbose=0
        )
        idx  = int(np.argmax(out_tokens[0, -1, :]))
        if idx == end_id:
            break
        word = tgt_tokenizer.index_word.get(idx, '')
        if word and word not in (START_TOKEN, END_TOKEN, PAD_TOKEN, UNK_TOKEN):
            decoded.append(word)
        target_seq[0, 0] = idx
        states = [h, c]

    return ' '.join(decoded)


print('✓ Inference functions defined')


In [ ]:
from tensorflow.keras.preprocessing.text import tokenizer_from_json


def verify_model_reload(output_dir: Path, direction: str, label: str,
                        n_samples: int = 5):
    """Reload model from disk and run a quick sanity-check."""
    print(f'\n{"="*70}')
    print(f'VERIFICATION: {label}')
    print(f'{"="*70}')

    model = keras.models.load_model(
        output_dir / 'model.keras', compile=False
    )
    with open(output_dir / 'src_tokenizer.json', encoding='utf-8') as f:
        src_tok = tokenizer_from_json(f.read())
    with open(output_dir / 'tgt_tokenizer.json', encoding='utf-8') as f:
        tgt_tok = tokenizer_from_json(f.read())

    print(f'  ✓ Model loaded: {model.count_params():,} parameters')
    print(f'  ✓ src_tokenizer: {len(src_tok.word_index)} tokens, '
          f'char_level={src_tok.char_level}')
    print(f'  ✓ tgt_tokenizer: {len(tgt_tok.word_index)} tokens')

    enc_model, dec_model = build_inference_models(model)
    print('  ✓ Inference models built')

    test_pairs = load_data(direction, 'test')
    print(f'\n  Testing first {n_samples} examples:')
    for i, pair in enumerate(test_pairs[:n_samples], 1):
        src_seq = src_tok.texts_to_sequences([pair['source']])
        src_seq = pad_sequences(src_seq, maxlen=MAX_SOURCE_LEN,
                                padding='post', truncating='post')
        pred = decode_sequence(src_seq, enc_model, dec_model, tgt_tok)
        match = '✓' if pred.strip() == pair['target'].strip().lower() else '✗'
        print(f'    {i}. {match} "{pair["source"]}" → "{pred}"')
        print(f'         expected: "{pair["target"]}"')


def evaluate_test_set(result: dict) -> dict:
    """Run greedy decoding on the full test set and report accuracy."""
    test_pairs    = result['test_pairs']       # fixed: was KeyError in original
    src_tokenizer = result['src_tokenizer']
    tgt_tokenizer = result['tgt_tokenizer']
    enc_model     = result['encoder_model']
    dec_model     = result['decoder_model']
    output_dir    = result['output_dir']

    correct, results = 0, []
    for pair in test_pairs:
        src_seq = src_tokenizer.texts_to_sequences([pair['source']])
        src_seq = pad_sequences(src_seq, maxlen=MAX_SOURCE_LEN,
                                padding='post', truncating='post')
        pred    = decode_sequence(src_seq, enc_model, dec_model, tgt_tokenizer)
        ok      = pred.strip() == pair['target'].strip().lower()
        if ok:
            correct += 1
        results.append({'source': pair['source'], 'target': pair['target'],
                         'predicted': pred, 'correct': ok})

    accuracy = correct / len(test_pairs) if test_pairs else 0.0
    out = {'accuracy': accuracy, 'correct': correct,
           'total': len(test_pairs), 'results': results}

    with open(output_dir / 'test_evaluation.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    return out


print('✓ Verification and evaluation functions defined')


In [ ]:
def train_model(direction: str, label: str) -> dict:
    """
    Train a single direction-specific model.
    Returns a result dict compatible with verify_model_reload
    and evaluate_test_set.
    """
    print('\n' + '='*70)
    print(f'TRAINING: {label}')
    print('='*70)
    start_time = time.time()

    # 1. Load data
    print('\n1. Loading data...')
    train_pairs = load_data(direction, 'train')
    val_pairs   = load_data(direction, 'val')
    test_pairs  = load_data(direction, 'test')
    print(f'   Train: {len(train_pairs)}  Val: {len(val_pairs)}  '
          f'Test: {len(test_pairs)}')

    # 2. Tokenizers
    print('\n2. Building tokenizers (train data only)...')
    src_tok, tgt_tok = prepare_tokenizers(train_pairs)
    src_vocab = max(src_tok.word_index.values()) + 1
    tgt_vocab = max(tgt_tok.word_index.values()) + 1
    print(f'   Source vocab: {src_vocab}  |  Target vocab: {tgt_vocab}')
    print(f'   START id: {tgt_tok.word_index.get(START_TOKEN)}  '
          f'END id: {tgt_tok.word_index.get(END_TOKEN)}')

    # 3. Sequences
    print('\n3. Encoding sequences...')
    tr_enc, tr_dec_in, tr_dec_tgt = prepare_sequences(
        train_pairs, src_tok, tgt_tok)
    vl_enc, vl_dec_in, vl_dec_tgt = prepare_sequences(
        val_pairs, src_tok, tgt_tok)

    # 4. Model
    print('\n4. Creating model...')
    model = create_model(src_vocab, tgt_vocab)
    model.compile(
        optimizer='adam',
        loss=create_masked_loss(),
        metrics=[create_masked_accuracy()]
    )
    print(f'   Parameters: {model.count_params():,}')

    # 5. Output dir
    out_dir = OUTPUT_DIR / direction
    out_dir.mkdir(parents=True, exist_ok=True)

    # 6. Callbacks
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(out_dir / 'model_best.weights.h5'),
            save_weights_only=True, monitor='val_loss',
            save_best_only=True, verbose=0
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=0
        ),
    ]

    # 7. Train
    print(f'\n5. Training (max {MAX_EPOCHS} epochs, patience={PATIENCE})...')
    history = model.fit(
        [tr_enc, tr_dec_in], tr_dec_tgt,
        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        validation_data=([vl_enc, vl_dec_in], vl_dec_tgt),
        callbacks=callbacks,
        verbose=2,
    )
    elapsed = time.time() - start_time

    # 8. Reload best weights
    model.load_weights(out_dir / 'model_best.weights.h5')

    # 9. Save
    print('\n6. Saving model and tokenizers...')
    model.save(out_dir / 'model.keras')
    save_tokenizer(src_tok, out_dir / 'src_tokenizer.json')
    save_tokenizer(tgt_tok, out_dir / 'tgt_tokenizer.json')

    config = {
        'direction': direction, 'label': label,
        'max_source_len': MAX_SOURCE_LEN,
        'max_target_len': MAX_TARGET_LEN,
        'embedding_dim': EMBEDDING_DIM, 'lstm_units': LSTM_UNITS,
        'src_vocab_size': src_vocab, 'tgt_vocab_size': tgt_vocab,
        'train_pairs': len(train_pairs), 'val_pairs': len(val_pairs),
        'test_pairs': len(test_pairs),
        'epochs_trained': len(history.history['loss']),
        'training_time_seconds': elapsed,
        'best_val_loss': float(min(history.history['val_loss'])),
    }
    with open(out_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    history_dict = {k: [float(v) for v in vs]
                    for k, vs in history.history.items()}
    with open(out_dir / 'training_history.json', 'w') as f:
        json.dump(history_dict, f, indent=2)

    print(f'\n✓ Done in {elapsed/60:.1f} min  '
          f'({len(history.history["loss"])} epochs)  '
          f'best val_loss={min(history.history["val_loss"]):.4f}')
    print(f'   Saved to: {out_dir}')

    # Build inference models so the result dict can be used directly
    enc_model, dec_model = build_inference_models(model)

    return {
        'model': model,
        'src_tokenizer': src_tok,
        'tgt_tokenizer': tgt_tok,
        'encoder_model': enc_model,
        'decoder_model': dec_model,
        'test_pairs': test_pairs,      # fixed: was missing in original
        'config': config,
        'history': history_dict,
        'output_dir': out_dir,
    }


print('✓ train_model function defined')


In [ ]:
all_results = {}
total_start = time.time()

for direction, label in DIRECTIONS:
    try:
        result = train_model(direction, label)
        all_results[direction] = result

        eval_out = evaluate_test_set(result)
        print(f'   Test exact-match accuracy: '
              f'{eval_out["accuracy"]*100:.1f}% '
              f'({eval_out["correct"]}/{eval_out["total"]})')

    except Exception as e:
        import traceback
        print(f'\n❌ {label} failed: {e}')
        traceback.print_exc()

total_time = time.time() - total_start
print(f'\n\nAll directions done in {total_time/3600:.2f} hours')


In [ ]:
for direction, label in DIRECTIONS:
    out_dir = OUTPUT_DIR / direction
    if not (out_dir / 'model.keras').exists():
        print(f'Skipping {label} — model not found')
        continue
    try:
        verify_model_reload(out_dir, direction, label)
    except Exception as e:
        print(f'❌ Verification error for {label}: {e}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training History — All Models', fontsize=14)

for idx, (direction, label) in enumerate(DIRECTIONS):
    hist_file = OUTPUT_DIR / direction / 'training_history.json'
    if not hist_file.exists():
        continue
    with open(hist_file) as f:
        h = json.load(f)

    ax = axes[idx // 2][idx % 2]
    ax.plot(h['loss'],     label='train loss')
    ax.plot(h['val_loss'], label='val loss')
    ax.set_title(label)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=120)
plt.show()
print(f'\nCurves saved to {OUTPUT_DIR / "training_curves.png"}')


In [ ]:
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
for direction, label in DIRECTIONS:
    cfg_file = OUTPUT_DIR / direction / 'config.json'
    if not cfg_file.exists():
        continue
    with open(cfg_file) as f:
        cfg = json.load(f)
    eval_file = OUTPUT_DIR / direction / 'test_evaluation.json'
    acc = '?'
    if eval_file.exists():
        with open(eval_file) as f:
            acc = f'{json.load(f)["accuracy"]*100:.1f}%'
    print(f'  {label:<25}  '
          f'{cfg["epochs_trained"]:>3} epochs  '
          f'val_loss={cfg["best_val_loss"]:.4f}  '
          f'test_acc={acc}')
print(f'\nModels saved to: {OUTPUT_DIR}')
